## Menghubungkan Google Drive dan Membaca Data Kasus

In [1]:
import os
import json
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split

# Mount Google Drive
drive.mount('/content/drive')

# Konfigurasi Path Direktori
BASE_DRIVE_DIR = '/content/drive/MyDrive/CBR/'
PROCESSED_CSV_PATH = os.path.join(BASE_DRIVE_DIR, 'data/processed/cases.csv')
RESULTS_DIR = os.path.join(BASE_DRIVE_DIR, 'data/results/')

os.makedirs(RESULTS_DIR, exist_ok=True)

# Load Data Utama
df_cases = pd.read_csv(PROCESSED_CSV_PATH)

# Memisahkan kembali Train dan Test dengan random_state yang SAMA seperti Tahap 3
df_train, df_test = train_test_split(df_cases, test_size=0.20, random_state=42)

print(f"Data Berhasil Disinkronkan!")
print(f"- Case Base (Train Data): {len(df_train)} dokumen")
print(f"- Kasus Baru (Test Data) : {len(df_test)} dokumen")

Mounted at /content/drive
Data Berhasil Disinkronkan!
- Case Base (Train Data): 24 dokumen
- Kasus Baru (Test Data) : 6 dokumen


## Membangun Mesin Pencari Internal (Retrieval Engine)

In [2]:


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix_train = tfidf_vectorizer.fit_transform(df_train['ringkasan_fakta'].fillna(''))

def retrieve(query: str, k: int = 5):
    query_vector = tfidf_vectorizer.transform([query])
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix_train).flatten()
    top_k_indices = np.argsort(similarity_scores)[::-1][:k]

    results = []
    for idx in top_k_indices:
        results.append({
            "case_id": int(df_train.iloc[idx]['case_id']),
            "solusi_amar": str(df_train.iloc[idx]['solusi_amar']),
            "similarity_score": float(similarity_scores[idx])
        })
    return results

print("Fungsi pencarian internal siap mendukung proses peminjaman solusi (Reuse).")

Fungsi pencarian internal siap mendukung proses peminjaman solusi (Reuse).


## Penerapan Algoritma Peminjaman Solusi (Case Reuse)

In [3]:


def predict_outcome(query: str, k: int = 5) -> str:
    """
    Mengambil amar putusan dari top-k kasus lama dan menentukan solusi akhir.
    """
    # Menjalankan fungsi retrieval untuk mendapat top-k kasus
    top_k_cases = retrieve(query, k=k)

    if not top_k_cases:
        return "Solusi tidak ditemukan."

    # Menerapkan Algoritma Weighted Similarity / Voting sederhana
    # Mengambil solusi dari kasus yang memiliki skor kemiripan tertinggi (Rank 1)  sebagai solusi rekomendasi yang paling logis dan berbobot

    best_match_case = top_k_cases[0]
    predicted_solution = best_match_case["solusi_amar"]

    # Berikan fallback tulisan jika kolom amar putusan ternyata kosong
    if pd.isna(predicted_solution) or predicted_solution.strip() == "" or predicted_solution == "Tidak Terdeteksi":
        predicted_solution = "Gugatan dikabulkan sebagian karena Tergugat terbukti wanprestasi."

    return predicted_solution

print("Fungsi predict_outcome() berhasil dibuat.")

Fungsi predict_outcome() berhasil dibuat.


## Simulasi Prediksi dan Ekspor Hasil

In [4]:
predictions_records = []

print("Memulai simulasi prediksi solusi atas 5 Kasus Baru (Data Test)...\n" + "="*70)

for i, row in df_test.head(5).iterrows():
    fakta_text = str(row['ringkasan_fakta']) if pd.notna(row['ringkasan_fakta']) else "Perkara gugatan wanprestasi."
    query_text = fakta_text[:500]

    # Mengambil top 5 id kasus termirip untuk dicatat di laporan hasil
    top_k_res = retrieve(query_text, k=5)
    top_5_ids = [c["case_id"] for c in top_k_res]

    # Jalankan fungsi prediksi solusi
    predicted_sol = predict_outcome(query_text, k=5)

    record = {
        "query_id": int(row['case_id']),
        "top_5_case_ids": str(top_5_ids),
        "predicted_solution": predicted_sol[:500]
    }
    predictions_records = [record] + predictions_records

    print(f"Kasus ID: {row['case_id']} -> Selesai diprediksi. Top-5 Kasus Mirip: {top_5_ids}")

# Mengubah hasil ke Pandas DataFrame
df_predictions = pd.DataFrame(predictions_records)

# Menyimpan file output sesuai aturan spesifikasi tugas
csv_predictions_path = os.path.join(RESULTS_DIR, 'predictions.csv')
df_predictions.to_csv(csv_predictions_path, index=False, encoding='utf-8')

print("="*70)
print("TAHAP 4 SELESAI")
print(f"File Hasil Prediksi Berhasil Disimpan di: {csv_predictions_path}")
print("\nPreview Hasil Prediksi Solusi:")
display(df_predictions.head(3))

Memulai simulasi prediksi solusi atas 5 Kasus Baru (Data Test)...
Kasus ID: 31 -> Selesai diprediksi. Top-5 Kasus Mirip: [10, 25, 32, 24, 30]
Kasus ID: 19 -> Selesai diprediksi. Top-5 Kasus Mirip: [8, 29, 33, 11, 6]
Kasus ID: 27 -> Selesai diprediksi. Top-5 Kasus Mirip: [29, 33, 1, 11, 17]
Kasus ID: 21 -> Selesai diprediksi. Top-5 Kasus Mirip: [8, 1, 17, 15, 28]
Kasus ID: 12 -> Selesai diprediksi. Top-5 Kasus Mirip: [23, 20, 1, 28, 30]
TAHAP 4 SELESAI
File Hasil Prediksi Berhasil Disimpan di: /content/drive/MyDrive/CBR/data/results/predictions.csv

Preview Hasil Prediksi Solusi:


,query_id,top_5_case_ids,predicted_solution
0,12,"[23, 20, 1, 28, 30]",Gugatan dikabulkan untuk sebagian dan menyatak...
1,21,"[8, 1, 17, 15, 28]",Gugatan dikabulkan untuk sebagian dan menyatak...
2,27,"[29, 33, 1, 11, 17]",Gugatan dikabulkan untuk sebagian dan menyatak...
